# Comparing Cost Functions

In this notebook, we take a look at the different fitting cost functions offered in PyBOP. Cost functions for fitting problems conventionally describe the distance between two points (the target and the prediction) which is to be minimised via PyBOP's optimisation algorithms.

### Setting up the Environment

If you don't already have PyBOP installed, check out the [installation guide](https://pybop-docs.readthedocs.io/en/latest/installation.html) first.

We begin by importing the necessary libraries. Let's also fix the random seed to generate consistent output during development.

In [ ]:
import numpy as np
import pybamm

import pybop

pybop.plot.use_backend("plotly")
go = pybop.plot.backends.PlotlyManager().go
pybop.plot.backends.PlotlyManager().pio.renderers.default = "notebook_connected"

np.random.seed(8)  # users can remove this line

For this notebook, we need to construct parameters, a model and a simulator before we can compare differing cost functions. This an arbitrary selection.

In [ ]:
model = pybamm.lithium_ion.SPM()
parameter_values = pybamm.ParameterValues("Chen2020")

Next, as we will need reference data to compare our model predictions to (via the cost function), we will create synthetic data from the model constructed above. 

In [ ]:
time = np.arange(0, 300, 20)
solution = pybop.pybamm.Simulator(
    model, parameter_values=parameter_values, protocol=time
).solve()
dataset = pybop.import_pybamm_solution(solution)

Now, we can construct parameters and bring this all together in the simulator class.

In [ ]:
parameter_values.update(
    {
        "Positive electrode thickness [m]": pybop.Parameter(
            pybop.Gaussian(7.56e-05, 0.5e-05, truncated_at=[65e-06, 10e-05]),
        ),
        "Positive particle radius [m]": pybop.Parameter(
            pybop.Gaussian(5.22e-06, 0.5e-06, truncated_at=[2e-06, 9e-06]),
        ),
    }
)

submesh_types = model.default_submesh_types
submesh_types.update(
    {
        "positive electrode": pybamm.SymbolicUniform1DSubMesh,
        "positive particle": pybamm.SymbolicUniform1DSubMesh,
    }
)

simulator = pybop.pybamm.Simulator(
    model,
    parameter_values=parameter_values,
    protocol=dataset,
    output_variables=["Voltage [V]"],
    submesh_types=submesh_types,
)

We can now generate a solution for some candidate parameter values.

In [ ]:
inputs = simulator.parameters.to_dict([7.56e-05, 5.22e-06])
solution = simulator.solve(inputs)

## Sum of squared errors and root mean squared error

First, let's start with two commonly-used cost functions: the sum of squared errors (SSE) and the root mean squared error (RMSE). Constructing these classes is very concise in PyBOP, and only requires the dataset.

In [ ]:
SSE = pybop.SumSquaredError(dataset)
RMSE = pybop.RootMeanSquaredError(dataset)

Now, we can investigate how these functions differ for the same dataset and inputs. To acquire the cost value for each cost function, we can use the `evaluate` method of the constructed cost class.

In [ ]:
SSE_value = SSE.evaluate(solution, inputs).get_values()
RMSE_value = RMSE.evaluate(solution, inputs).get_values()
print(SSE_value, RMSE_value)

[6.57773002e-11] [2.09407577e-06]


We can also generate a random sample of candidate solutions from the parameters, if the parameters have a prior distribution from which to sample.

In [ ]:
samples = simulator.parameters.sample_from_distribution(2)
print("Samples:", samples)
inputs = [simulator.parameters.to_dict(s) for s in samples]
solution = simulator.solve(inputs)
print("SSE costs:", SSE.evaluate(solution, inputs).get_values())
print("RMSE costs:", RMSE.evaluate(solution, inputs).get_values())

Samples: [[8.13658824e-05 5.78129588e-06]
 [8.49368515e-05 5.25871051e-06]]


SSE costs: [1.16563713e-05 8.49861618e-04]
RMSE costs: [0.00088153 0.00752711]


### Comparing RMSE and SSE

Now, let's vary one of the parameters, and keep a fixed value for the other, to create a plot comparing the cost values for the RMSE and SSE functions.

In [ ]:
x_range = np.linspace(4.72e-06, 5.72e-06, 25)
inputs = [simulator.parameters.to_dict([7.56e-05, i]) for i in x_range]
solution = simulator.solve(inputs)
y_SSE = SSE.evaluate(solution, inputs).get_values()
y_RMSE = RMSE.evaluate(solution, inputs).get_values()

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_range, y=y_SSE, mode="lines", name="SSE"))
fig.add_trace(go.Scatter(x=x_range, y=y_RMSE, mode="lines", name="RMSE"))
fig.show()

In this situation, it's clear that the curvature of the SSE cost is greater than that of the RMSE. This can improve the rate of convergence for certain optimisation algorithms. However, with incorrect hyperparameter values, larger gradients can also result in the algorithm not converging due to sampling locations outside of the "cost valley", e.g. infeasible parameter values.

## Minkowski distance

Next, let's investigate the Minkowski distance. The Minkowski cost takes a general form, which allows for hyperparameter calibration on the cost function itself, given by

$\mathcal{L_p} = \displaystyle \Big(\sum_i  |\hat{y_i}-y_i|^p\Big)^{1/p}$

where $p ≥ 0$ is the order of the Minkowski distance.

For $p = 1$, it is the Manhattan distance.  
For $p = 2$, it is the Euclidean distance.  
For $p ≥ 1$, the Minkowski distance is a metric, but for $0<p<1$, note that the Minkowski distance is not a metric because the triangle inequality does not hold [[1]](https://en.wikipedia.org/wiki/Minkowski_distance), [[2]](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.minkowski.html#scipy.spatial.distance.minkowski).

The Minkowski distance is related to the previous two cost function as follows. The RMSE can be computed by dividing the Euclidean distance by the square root of the length of the target vector, while the SSE is the square of the Euclidean distance.

PyBOP offers a Minkowski class, which we will construct below. This class has an optional argument of `p` which designates the order in the above equation. This value can be a float, with the only requirement that it is not negative. First, let's confirm the relationship between the SSE, RMSE and the Minkowski distance with a `p` value of 2.

In [ ]:
minkowski = pybop.Minkowski(dataset, p=2)
y_minkowski = minkowski.evaluate(solution, inputs).get_values()

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x_range,
        y=np.asarray(y_RMSE) * np.sqrt(len(time)),
        mode="lines",
        name="RMSE*N",
    )
)
fig.add_trace(
    go.Scatter(
        x=x_range,
        y=np.sqrt(y_SSE),
        mode="lines",
        line=dict(dash="dash"),
        name="sqrt(SSE)",
    )
)
fig.add_trace(
    go.Scatter(
        x=x_range, y=y_minkowski, mode="lines", line=dict(dash="dot"), name="Minkowski"
    )
)
fig.show()

As expected, these lines lie on top of one another. Now, let's take a look at how the Minkowski cost changes for different orders, `p`.

In [ ]:
p_orders = np.append(0.75, np.linspace(1, 3, 3))
y_minkowski_p = []
for p in p_orders:
    minkowski_p = pybop.Minkowski(dataset, p=p)
    y_minkowski_p.append(minkowski_p.evaluate(solution, inputs).get_values())

fig = go.Figure()
for i, p in enumerate(p_orders):
    fig.add_trace(
        go.Scatter(x=x_range, y=y_minkowski_p[i], mode="lines", name=f"Minkowski {p}")
    )
fig.update_yaxes(range=[0, np.max(y_minkowski_p[2])])
fig.show()

As seen above, the Minkowski cost allows for a range of different cost functions to be created. This provides users with another hyperparameter to calibrate for optimisation algorithm convergence. This addition does expand the global search space, and should be carefully considered before deciding upon.

## Sum of power

Next, we introduce a similar cost function, the `SumOfPower` implementation. This cost function is the $p$-th power of the Minkowski distance of order $p$. It provides a generalised formulation for the Sum of Squared Errors (SSE) cost function, and is given by,

$\mathcal{L_p} = \displaystyle \sum_i  |\hat{y_i}-y_i|^p$

where $p ≥ 0$ is the power order. A few special cases include,

$p = 1$: Sum of Absolute Differences
$p = 2$: Sum of Squared Differences

Next we repeat the above examples with the `SumOfPower` class.

In [ ]:
sumofpower = pybop.SumOfPower(dataset, p=2)
y_sumofpower = sumofpower.evaluate(solution, inputs).get_values()

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x_range,
        y=np.asarray(y_RMSE) * np.sqrt(len(time)),
        mode="lines",
        name="RMSE*N",
    )
)
fig.add_trace(
    go.Scatter(
        x=x_range,
        y=y_SSE,
        mode="lines",
        line=dict(dash="dash"),
        name="SSE",
    )
)
fig.add_trace(
    go.Scatter(
        x=x_range,
        y=y_sumofpower,
        mode="lines",
        line=dict(dash="dot"),
        name="Sum of Power",
    )
)
fig.show()

As expected, the `SumOfPower` with order `p=2` equates to the `SSE` implementation. Next, we compare the `Minkowski` to the `SumOfPower`,

In [ ]:
p_orders = np.asarray([0.75, 2])
y_minkowski_p = []
y_sumofpower_p = []
for p in p_orders:
    minkowski_p = pybop.Minkowski(dataset, p=p)
    y_minkowski_p.append(minkowski_p.evaluate(solution, inputs).get_values())

    sumofpower_p = pybop.SumOfPower(dataset, p=p)
    y_sumofpower_p.append(sumofpower_p.evaluate(solution, inputs).get_values())

fig = go.Figure()
for i, p in enumerate(p_orders):
    fig.add_trace(
        go.Scatter(x=x_range, y=y_minkowski_p[i], mode="lines", name=f"Minkowski {p}")
    )
    fig.add_trace(
        go.Scatter(
            x=x_range,
            y=y_sumofpower_p[i],
            mode="lines",
            line=dict(dash="dash"),
            name=f"Sum of Power {p}",
        )
    )
fig.update_yaxes(range=[0, 0.3])
fig.show()

The figure demonstrates the distinct behaviour of the `Minkowski` distance and the `SumOfPower` function. One notable difference is the effect of the `1/p` exponent in the `Minkowski` distance, which has a linearising impact on the response. This linearisation can enhance the robustness of certain optimisation algorithms, potentially making them less sensitive to outliers or extreme values. However, this increased robustness may come at the cost of a slower convergence rate, as the linearised response might require more iterations to reach the optimal solution. In contrast, the `SumOfPower` function does not exhibit this linearising effect, which can lead to faster convergence in some cases but may be more susceptible to the influence of outliers or extreme values.

## Concluding thoughts

In this notebook, we've shown the different fitting cost functions offered in PyBOP. Selection between these functions can affect the optimisation result in the case that the optimiser hyperparameter values are not properly calibrated. 